# Обработка r3d и визуализация


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import io
import json
import plistlib
import zipfile

import lzfse
import numpy as np
from PIL import Image
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"


In [ ]:
@dataclass
class SingleFrameConfig:
    r3d_path: str
    frame_index: int = 0
    confidence_threshold: int = 1
    z_min: float | None = None
    z_max: float | None = None
    sample_step: int = 1
    max_points_plot: int = 35000
    point_size: float = 2.5
    use_rgb: bool = True
    plot_height: int = 800


def parse_metadata(raw: bytes):
    stripped = raw.lstrip()
    if stripped.startswith((b"{", b"[")):
        return json.loads(raw.decode("utf-8", errors="ignore"))
    return plistlib.loads(raw)


def scan_r3d_file(r3d_path: str):
    zf = zipfile.ZipFile(r3d_path, "r")
    names = zf.namelist()
    metadata_candidates = []
    for name in names:
        file_name = Path(name).name.lower()
        if "metadata" in file_name:
            metadata_candidates.append(name)

    metadata_candidates.sort(key=len)
    metadata_name = metadata_candidates[0]
    metadata = parse_metadata(zf.read(metadata_name))

    frame_map = {}
    for name in names:
        pth = Path(name)
        suffix = pth.suffix.lower()
        if suffix in {".jpg", ".jpeg", ".depth", ".conf"}:
            frame_map.setdefault(pth.stem, {})[suffix] = name

    frame_ids = sorted(
        [frame_id for frame_id, members in frame_map.items() if ".depth" in members],
        key=lambda value: (0, int(value)) if value.isdigit() else (1, value),
    )

    return zf, metadata, frame_map, frame_ids


def select_cam_matrix(cam_matrix_values, rgb_w: int, rgb_h: int):
    cam_matrix_values = np.asarray(cam_matrix_values, dtype=np.float64).reshape(-1)
    candidates = [
        cam_matrix_values.reshape(3, 3),
        cam_matrix_values.reshape(3, 3, order="F"),
    ]

    def score(cam_matrix):
        fx, fy, cx, cy = cam_matrix[0, 0], cam_matrix[1, 1], cam_matrix[0, 2], cam_matrix[1, 2]
        return (
            float(np.isfinite(cam_matrix).all())
            + 2 * (abs(cam_matrix[2, 2] - 1.0) < 1e-3)
            + 2 * (abs(cam_matrix[2, 0]) < 1e-3 and abs(cam_matrix[2, 1]) < 1e-3)
            + 1 * (abs(cam_matrix[0, 1]) < 1e-3 and abs(cam_matrix[1, 0]) < 1e-3)
            + 2 * (fx > 0 and fy > 0)
            + 2 * (0 <= cx <= rgb_w * 1.2)
            + 2 * (0 <= cy <= rgb_h * 1.2)
        )

    return max(candidates, key=score)


def cam_matrix_to_depth(rgb_cam_matrix: np.ndarray, rgb_shape_hw: tuple[int, int], depth_shape_hw: tuple[int, int]):
    rgb_h, rgb_w = rgb_shape_hw
    depth_h, depth_w = depth_shape_hw
    sx = depth_w / rgb_w
    sy = depth_h / rgb_h
    depth_cam_matrix = rgb_cam_matrix.copy().astype(np.float64)
    depth_cam_matrix[0, 0] *= sx
    depth_cam_matrix[1, 1] *= sy
    depth_cam_matrix[0, 2] *= sx
    depth_cam_matrix[1, 2] *= sy
    return depth_cam_matrix


def load_rgb_frame(zf: zipfile.ZipFile, member_name: str):
    raw_image = zf.read(member_name)
    image_buffer = io.BytesIO(raw_image)
    image = Image.open(image_buffer).convert("RGB")
    return np.asarray(image)


def load_depth_frame(zf: zipfile.ZipFile, member_name: str, depth_shape_hw: tuple[int, int]):
    raw = lzfse.decompress(zf.read(member_name))
    return np.frombuffer(raw, dtype="<f4").reshape(depth_shape_hw)


def load_conf_frame(zf: zipfile.ZipFile, member_name: str, depth_shape_hw: tuple[int, int]):
    raw = lzfse.decompress(zf.read(member_name))
    return np.frombuffer(raw, dtype=np.uint8).reshape(depth_shape_hw)


def resize_rgb_to_depth(rgb: np.ndarray, depth_shape_hw: tuple[int, int]):
    h, w = depth_shape_hw
    return np.asarray(Image.fromarray(rgb).resize((w, h), Image.BILINEAR))


def depth_to_point_cloud(
    depth: np.ndarray,
    depth_cam_matrix: np.ndarray,
    rgb: np.ndarray | None = None,
    confidence: np.ndarray | None = None,
    confidence_threshold: int = 0,
    z_min: float | None = None,
    z_max: float | None = None,
    sample_step: int = 1,
):
    h, w = depth.shape
    v, u = np.indices((h, w))
    z = depth.astype(np.float32)

    mask = np.isfinite(z) & (z > 0)
    if confidence is not None:
        mask &= confidence >= confidence_threshold
    if z_min is not None:
        mask &= z >= z_min
    if z_max is not None:
        mask &= z <= z_max
    if sample_step > 1:
        mask &= (u % sample_step == 0) & (v % sample_step == 0)

    u = u[mask].astype(np.float32)
    v = v[mask].astype(np.float32)
    z = z[mask].astype(np.float32)

    fx, fy = float(depth_cam_matrix[0, 0]), float(depth_cam_matrix[1, 1])
    cx, cy = float(depth_cam_matrix[0, 2]), float(depth_cam_matrix[1, 2])

    x = (u - cx) * z / fx
    y = (v - cy) * z / fy
    points = np.stack([x, y, z], axis=1)

    colors = None
    if rgb is not None:
        rgb_small = resize_rgb_to_depth(rgb, depth.shape)
        colors = rgb_small[mask].astype(np.uint8)

    return points, colors


def sample_points(points: np.ndarray, colors: np.ndarray | None, max_points_plot: int, seed: int = 42):
    if len(points) <= max_points_plot:
        return points, colors
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(points), size=max_points_plot, replace=False))
    color = None if colors is None else colors[idx]
    return points[idx], color


def plot_point_cloud(points: np.ndarray, colors: np.ndarray | None, max_points_plot: int, point_size: float, height: int, title: str):
    points, colors = sample_points(points, colors, max_points_plot=max_points_plot)

    if colors is None:
        marker = {
            "size": point_size,
            "color": points[:, 2],
            "colorscale": "Viridis",
            "opacity": 0.85,
            "colorbar": {"title": "Z, m"},
        }
    else:
        rgb_colors = [
            f"rgb({r},{g},{b})"
            for r, g, b in colors.tolist()
        ]
        marker = {
            "size": point_size,
            "color": rgb_colors,
            "opacity": 0.85,
        }
    
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points[:, 0],
                y=points[:, 1],
                z=points[:, 2],
                mode="markers",
                marker=marker,
                hovertemplate="x = %{x:.3f} m<br>y = %{y:.3f} m<br>z = %{z:.3f} m<extra></extra>",
            )
        ]
    )
    fig.update_layout(
        title=title,
        height=height,
        scene=dict(
            xaxis_title="X, m",
            yaxis_title="Y, m",
            zaxis_title="Z, m",
            aspectmode="data",
            dragmode="orbit",
            camera=dict(up=dict(x=0, y=-1, z=0), eye=dict(x=1.25, y=-1.25, z=0.9)),
        ),
        margin=dict(l=0, r=0, t=45, b=0),
        showlegend=False,
    )
    return fig


def visualize_single_frame(config: SingleFrameConfig):
    zf, metadata, frame_map, frame_ids = scan_r3d_file(config.r3d_path)

    depth_shape = (int(metadata["dh"]), int(metadata["dw"]))
    rgb_shape = (int(metadata["h"]), int(metadata["w"]))
    rgb_cam_matrix = select_cam_matrix(metadata["K"], rgb_w=rgb_shape[1], rgb_h=rgb_shape[0])
    depth_cam_matrix = cam_matrix_to_depth(rgb_cam_matrix, rgb_shape, depth_shape)

    frame_id = frame_ids[config.frame_index]
    members = frame_map[frame_id]
    rgb_name = members.get(".jpg")
    if (rgb_name):
        rgb = load_rgb_frame(zf, rgb_name)
    else:
        rgb = None
    depth = load_depth_frame(zf, members[".depth"], depth_shape)
    confidence = load_conf_frame(zf, members[".conf"], depth_shape) if ".conf" in members else None

    points, colors = depth_to_point_cloud(
        depth=depth,
        depth_cam_matrix=depth_cam_matrix,
        rgb=rgb if config.use_rgb else None,
        confidence=confidence,
        confidence_threshold=config.confidence_threshold,
        z_min=config.z_min,
        z_max=config.z_max,
        sample_step=config.sample_step,
    )

    fig = plot_point_cloud(
        points=points,
        colors=colors,
        max_points_plot=config.max_points_plot,
        point_size=config.point_size,
        height=config.plot_height,
        title=f"Single frame point cloud",
    )
    fig.show(config={"scrollZoom": True, "modeBarButtonsToAdd": ["orbitRotation", "resetCameraDefault3d"]})

    zf.close()
    return {"frame_id": frame_id, "points": points, "colors": colors, "figure": fig}


In [ ]:
R3D_PATH = "2026-03-24--23-18-48.r3d"

config = SingleFrameConfig(
    r3d_path=R3D_PATH,
    frame_index=0,
    confidence_threshold=1,
    z_min=None,
    z_max=None,
    sample_step=1,
    max_points_plot=35000,
    point_size=2.5,
    use_rgb=True,
    plot_height=800,
)

single_frame = visualize_single_frame(config)
single_frame